# **Ventana de maduración de 15 meses: análisis y figuras.**

Este cuaderno regenera, por sí solo, todo el soporte del **Anexo A (ventana de maduración)** de la bitácora y de la ficha de documentación: el contraste de la fecha de default, la curva de la LGD (primer intento, descartado), la curva de recuperación con N = 15, la sensibilidad de N en 9 combinaciones, la robustez por plazo, la recuperación por calificación y el efecto de la ventana por año de originación.

Las figuras se guardan en 300 dpi y las tablas en CSV en la carpeta `Anexo_Ventana_Maduracion`, que al final se descarga comprimida.

## **1. Carga de la base y población.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import shutil

In [ ]:
# Estilo de las figuras y carpeta de salida
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})
AZUL, GRIS, ROJO = '#1f4e79', '#9A9A9A', '#c0392b'
carpeta_salida = 'Anexo_Ventana_Maduracion'
os.makedirs(carpeta_salida, exist_ok=True)

def guardar(fig, nombre):
    fig.savefig(os.path.join(carpeta_salida, nombre), bbox_inches='tight')
    plt.show()

In [ ]:
traduccion = {'id': 'id_prestamo', 'loan_status': 'estado', 'funded_amnt': 'monto', 'total_rec_prncp': 'capital_pagado',
              'recoveries': 'recuperacion', 'collection_recovery_fee': 'comision', 'grade': 'calificacion', 'term': 'plazo',
              'issue_d': 'fecha_originacion', 'last_pymnt_d': 'fecha_ultimo_pago', 'installment': 'cuota', 'total_pymnt': 'total_pagado'}

In [ ]:
# Fuente de datos, en este orden: 1) copia versionada en el Drive del equipo; 2) copia publicada en el Release
# v1.0-avance3 de GitHub (cualquier persona puede descargarla); 3) descarga desde Kaggle.
# En todos los casos se verifica la huella SHA-256 registrada en Versionado_Base_Datos.md (descarga original: 26 de octubre de 2026).
RUTA_DRIVE = '/content/drive/MyDrive/LGD_Base_Versionada/lending_club_accepted_2007_2018Q4.parquet'
URL_RELEASE = 'https://github.com/mariacamilacc1/LGD-Happen-Bank/releases/download/v1.0-avance3/lending_club_accepted_2007_2018Q4.parquet'
RUTA_LOCAL = 'lending_club_accepted_2007_2018Q4.parquet'
SHA256_PARQUET = 'e964f05b0ad2024931e436630f586e9790dc47ff45242afa53860ec2b974a8d8'
SHA256_KAGGLE = '55c16f75120f897683f02e7aabcf080d0e4a20c4832feb1d592cfa941bd62a2d'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Sin Google Drive: se usará la copia publicada en GitHub o, si no está disponible, Kaggle')

In [ ]:
import hashlib
import urllib.request

def sha256(ruta, bloque=2 ** 20):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for parte in iter(lambda: f.read(bloque), b''):
            h.update(parte)
    return h.hexdigest()

archivos_verificados = set()

def verificar(ruta, esperado, nombre):
    if ruta in archivos_verificados:
        return
    if sha256(ruta) == esperado:
        print(f'Huella SHA-256 verificada: {nombre} coincide con la base del proyecto (descarga del 26 de octubre de 2026)')
    else:
        print(f'ATENCIÓN: la huella SHA-256 de {nombre} no coincide con la registrada; los resultados pueden diferir de los del proyecto')
    archivos_verificados.add(ruta)

# Carga las columnas pedidas desde la primera fuente disponible
def cargar_base(columnas):
    if os.path.exists(RUTA_DRIVE):
        verificar(RUTA_DRIVE, SHA256_PARQUET, 'la copia versionada de Google Drive')
        return pd.read_parquet(RUTA_DRIVE, columns=columnas)
    try:
        if not os.path.exists(RUTA_LOCAL):
            print('Descargando la copia versionada desde GitHub (59 MB)...')
            urllib.request.urlretrieve(URL_RELEASE, RUTA_LOCAL)
        verificar(RUTA_LOCAL, SHA256_PARQUET, 'la copia versionada de GitHub')
        return pd.read_parquet(RUTA_LOCAL, columns=columnas)
    except Exception as error:
        print(f'No se pudo usar la copia de GitHub ({error}); se descarga desde Kaggle')
    import kagglehub  # solo se necesita si no hay copia versionada disponible
    carpeta = kagglehub.dataset_download('wordsforthewise/lending-club')
    candidatos = [a for a in glob.glob(os.path.join(carpeta, '**', 'accepted_2007_to_2018*.csv*'), recursive=True) if os.path.isfile(a)]
    ruta = sorted(candidatos, key=lambda a: not a.endswith('.gz'))[0]
    verificar(ruta, SHA256_KAGGLE, 'el archivo de Kaggle')
    return pd.read_csv(ruta, usecols=columnas, dtype={'id': str}, low_memory=False)

In [ ]:
df = cargar_base(list(traduccion.keys()))
if df.shape[0] < 2_000_000:
    raise ValueError(f'Base incompleta: {df.shape[0]:,} filas')
df.shape

In [ ]:
df = df[pd.to_numeric(df['id'], errors='coerce').notna()].rename(columns=traduccion).reset_index(drop=True)
df['fecha_originacion'] = pd.to_datetime(df['fecha_originacion'], format='%b-%Y')
df['fecha_ultimo_pago'] = pd.to_datetime(df['fecha_ultimo_pago'], format='%b-%Y')
df['plazo_meses'] = df['plazo'].str.extract(r'(\d+)')[0].astype(float)

In [ ]:
# Población: créditos castigados (incluye fuera de política); se excluyen los 'Default'
df_def = df[df['estado'].str.contains('Charged Off', na=False)].copy()
df_def['lgd'] = ((df_def['monto'] - df_def['capital_pagado'] - (df_def['recuperacion'] - df_def['comision'])) / df_def['monto']).clip(0, 1)
df_def['anio_originacion'] = df_def['fecha_originacion'].dt.year
print(f'Créditos castigados: {len(df_def):,}')

## **2. Tiempo desde el default.**

La base no trae la fecha de default: se aproxima con la fecha del último pago y, si el crédito nunca pagó, con la de originación. Los meses desde el default se cuentan hasta la fecha de corte (el último pago registrado en toda la base).

In [ ]:
fecha_corte = df['fecha_ultimo_pago'].max()
mes = lambda f: f.dt.year * 12 + f.dt.month
df_def['fecha_default'] = df_def['fecha_ultimo_pago'].fillna(df_def['fecha_originacion'])
df_def['meses_desde_default'] = (fecha_corte.year * 12 + fecha_corte.month) - mes(df_def['fecha_default'])
print(f"Fecha de corte: {fecha_corte.strftime('%Y-%m')} | Créditos sin pagos: {df_def['fecha_ultimo_pago'].isna().sum():,}")

In [ ]:
# Contraste con la fecha estimada por cuotas pagadas: originación + (total pagado - recuperaciones) / cuota
cuotas_pagadas = ((df_def['total_pagado'] - df_def['recuperacion']) / df_def['cuota']).round().clip(0, df_def['plazo_meses'])
con_fecha = df_def['fecha_ultimo_pago'].notna()
diferencia = (mes(df_def['fecha_ultimo_pago']) - (mes(df_def['fecha_originacion']) + cuotas_pagadas))[con_fecha]
contraste = pd.Series({'Mismo mes (%)': (diferencia == 0).mean() * 100,
                       'Un mes o menos (%)': (diferencia.abs() <= 1).mean() * 100,
                       'Seis meses o menos (%)': (diferencia.abs() <= 6).mean() * 100}).round(1)
contraste.to_csv(os.path.join(carpeta_salida, 'tabla_contraste_fechas.csv'))
contraste

## **3. Curvas según los meses desde el default.**

Para cada mes se calcula el promedio y un promedio móvil de 6 meses ponderado por el número de créditos. La **tasa de recuperación neta** = (recuperación − comisión) / saldo al default se usa solo para elegir N; la fórmula de la LGD no cambia.

In [ ]:
saldo_default = df_def['monto'] - df_def['capital_pagado']
df_def['tasa_recuperacion'] = ((df_def['recuperacion'] - df_def['comision']) / saldo_default).where(saldo_default > 0).clip(0, 1)

def curva(datos, columna, max_mes=96):
    g = datos[datos['meses_desde_default'].between(0, max_mes)].groupby('meses_desde_default')[columna].agg(['count', 'sum'])
    g['media'] = g['sum'] / g['count']
    g['movil'] = g['sum'].rolling(6, min_periods=1).sum() / g['count'].rolling(6, min_periods=1).sum()
    return g

curva_lgd = curva(df_def, 'lgd', 72)
curva_rec = curva(df_def, 'tasa_recuperacion', 96)

In [ ]:
# Figura A1: curva de la LGD (primer intento, descartado)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(curva_lgd.index, curva_lgd['media'] * 100, color=GRIS, alpha=0.6, label='LGD media mensual')
ax.plot(curva_lgd.index, curva_lgd['movil'] * 100, color=AZUL, linewidth=2.5, label='Promedio móvil de 6 meses')
ax.set_title('La LGD media no se estabiliza con el tiempo desde el default')
ax.set_xlabel('Meses transcurridos desde el default')
ax.set_ylabel('LGD media (% del monto)')
ax.legend(frameon=False)
guardar(fig, 'figura_A1_curva_lgd.png')

## **4. Sensibilidad de N.**

N es el primer mes desde el cual el promedio móvil de la recuperación se mantiene dentro de la tolerancia alrededor de la recuperación de referencia, hasta el final del tramo. Se prueban 3 tramos de referencia (antes del quiebre del mes 66) y 3 tolerancias.

In [ ]:
filas = []
for inicio, fin in [(18, 60), (24, 60), (24, 54)]:
    referencia = df_def.loc[df_def['meses_desde_default'].between(inicio, fin), 'tasa_recuperacion'].mean()
    for tolerancia in [0.005, 0.01, 0.02]:
        n = next((m for m in range(0, fin + 1) if (abs(curva_rec.loc[m:fin, 'movil'] - referencia) <= tolerancia).all()), None)
        excluidos = (df_def['meses_desde_default'] < n).sum() if n is not None else np.nan
        filas.append({'Tramo de referencia (meses)': f'{inicio}–{fin}', 'Recuperación de referencia (%)': round(referencia * 100, 1),
                      'Tolerancia (pp)': tolerancia * 100, 'N (meses)': n if n is not None else 'No se encontró',
                      'Excluidos': excluidos, 'Excluidos (%)': round(excluidos / len(df_def) * 100, 1) if n is not None else np.nan})
sensibilidad = pd.DataFrame(filas)
sensibilidad.to_csv(os.path.join(carpeta_salida, 'tabla_sensibilidad_N.csv'), index=False)
sensibilidad

In [ ]:
# Decisión adoptada: N = 15 meses (fin del aumento de la recuperación, estable con ±2 pp en los tres tramos).
# N = 52 (tolerancia de ±1 pp) se descartó: excluye cerca del 91 % y crea un sesgo de selección.
N = 15
N_DESCARTADO = 52
referencia_18_60 = df_def.loc[df_def['meses_desde_default'].between(18, 60), 'tasa_recuperacion'].mean()
df_def['maduro'] = df_def['meses_desde_default'] >= N
print(f"N = {N} meses | excluidos: {(~df_def['maduro']).sum():,} ({(~df_def['maduro']).mean() * 100:.1f} %) | quedan: {df_def['maduro'].sum():,}")
print(f"Con N = {N_DESCARTADO} se excluirían: {(df_def['meses_desde_default'] < N_DESCARTADO).mean() * 100:.1f} %")

In [ ]:
# Figura A2: curva de recuperación con N = 15
fig, ax = plt.subplots(figsize=(10, 5))
ax.axvspan(0, N, color=GRIS, alpha=0.15, label=f'Excluidos por la ventana (< {N} meses)')
ax.axhspan((referencia_18_60 - 0.02) * 100, (referencia_18_60 + 0.02) * 100, color=AZUL, alpha=0.08, label='Referencia (18–60 meses) ± 2 pp')
ax.plot(curva_rec.index, curva_rec['media'] * 100, color=GRIS, alpha=0.6, label='Recuperación media mensual')
ax.plot(curva_rec.index, curva_rec['movil'] * 100, color=AZUL, linewidth=2.5, label='Promedio móvil de 6 meses')
ax.axvline(N, color='black', linestyle='--', linewidth=1)
ax.annotate(f'N = {N} meses', xy=(N, referencia_18_60 * 100 + 3), xytext=(N + 2, referencia_18_60 * 100 + 3))
ax.set_title(f'La recuperación se estabiliza a partir de los {N} meses desde el default')
ax.set_xlabel('Meses transcurridos desde el default')
ax.set_ylabel('Recuperación neta media (% del saldo al default)')
ax.legend(frameon=False, fontsize=9)
guardar(fig, 'figura_A2_curva_recuperacion.png')

## **5. Robustez de N.**

In [ ]:
# Figura A3: la curva de recuperación por plazo (36 y 60 meses)
fig, ax = plt.subplots(figsize=(10, 5))
for plazo, color in [(36, GRIS), (60, AZUL)]:
    c = curva(df_def[df_def['plazo_meses'] == plazo], 'tasa_recuperacion', 96)
    ax.plot(c.index, c['movil'] * 100, color=color, linewidth=2.5, label=f'Plazo de {plazo} meses')
ax.axvline(N, color='black', linestyle='--', linewidth=1)
ax.set_title('La recuperación se estabiliza en el mismo punto con los dos plazos')
ax.set_xlabel('Meses transcurridos desde el default')
ax.set_ylabel('Recuperación neta, promedio móvil (%)')
ax.legend(frameon=False)
guardar(fig, 'figura_A3_recuperacion_por_plazo.png')

In [ ]:
# Recuperación por calificación (créditos tras la ventana)
por_calificacion = df_def[df_def['maduro']].groupby('calificacion').agg(creditos=('lgd', 'size'),
                                                                       recuperacion_media=('tasa_recuperacion', 'mean'),
                                                                       lgd_media=('lgd', 'mean'))
por_calificacion[['recuperacion_media', 'lgd_media']] *= 100
por_calificacion.round(1).to_csv(os.path.join(carpeta_salida, 'tabla_por_calificacion.csv'))
por_calificacion.round(1)

In [ ]:
# Figura A4: la recuperación es parecida entre calificaciones
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(por_calificacion.index, por_calificacion['recuperacion_media'], color=AZUL)
ax.set_title('La recuperación del saldo es parecida en todas las calificaciones')
ax.set_xlabel('Calificación de riesgo')
ax.set_ylabel('Recuperación neta media (% del saldo)')
guardar(fig, 'figura_A4_recuperacion_por_calificacion.png')

## **6. Efecto de la ventana por año de originación.**

In [ ]:
por_anio = pd.DataFrame({
    'Castigados': df_def.groupby('anio_originacion').size(),
    'Después de la ventana (N = 15)': df_def[df_def['maduro']].groupby('anio_originacion').size(),
}).fillna(0).astype(int)
por_anio['% que queda'] = (por_anio['Después de la ventana (N = 15)'] / por_anio['Castigados'] * 100).round(1)
por_anio['LGD todos (%)'] = (df_def.groupby('anio_originacion')['lgd'].mean() * 100).round(1)
por_anio[f'LGD con N = {N} (%)'] = (df_def[df_def['maduro']].groupby('anio_originacion')['lgd'].mean() * 100).round(1)
por_anio[f'LGD con N = {N_DESCARTADO} (%)'] = (df_def[df_def['meses_desde_default'] >= N_DESCARTADO].groupby('anio_originacion')['lgd'].mean() * 100).round(1)
por_anio.to_csv(os.path.join(carpeta_salida, 'tabla_por_anio.csv'))
por_anio

In [ ]:
# Figura A5: LGD por año con todos los castigados, con N = 15 y con N = 52
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(por_anio.index, por_anio['LGD todos (%)'], marker='o', color=GRIS, label='Todos los castigados')
ax.plot(por_anio.index, por_anio[f'LGD con N = {N} (%)'], marker='o', color=AZUL, linewidth=2.5, label=f'Ventana de {N} meses (adoptada)')
ax.plot(por_anio.index, por_anio[f'LGD con N = {N_DESCARTADO} (%)'], marker='o', color=ROJO, linestyle='--', label=f'Ventana de {N_DESCARTADO} meses (descartada)')
ax.set_title('Con 52 meses la LGD de los años recientes se infla por sesgo de selección')
ax.set_xlabel('Año de originación')
ax.set_ylabel('LGD media (%)')
ax.legend(frameon=False)
guardar(fig, 'figura_A5_lgd_por_anio.png')

## **7. Descarga de las figuras y tablas.**

In [ ]:
shutil.make_archive(carpeta_salida, 'zip', carpeta_salida)
print(sorted(os.listdir(carpeta_salida)))

In [ ]:
# En Colab descarga el .zip con todas las figuras (300 dpi) y tablas
from google.colab import files
files.download(f'{carpeta_salida}.zip')